### Train Best Model with Optimized Parameters

Uses optimized parameters for max returns + min losses:
- gamma: 0.89 (9-week horizon)
- softmax_temperature: 0.9 (decisive actions)
- rolling_vol_window: 10 (quick volatility reaction)


In [1]:

from train import train
from config import get_config
import numpy as np
from pathlib import Path

# Ensure correct data directory path
DATA_DIR = '../data_hierarchical'


n_runs = 10
print("="*70)
print(f"RUNNING {n_runs} TIMES TO GET AVERAGE PERFORMANCE")
print("="*70)

best_params = {
    'gamma': 0.85,                    # n-week horizon - optimal for trends + quick loss cuts
    'softmax_temperature': 1.8,       # Decisive actions, not exploratory
    'rolling_vol_window': 12,         # Quick volatility reaction
    'transaction_cost': 0.001,
    'total_steps': 500_000,
    'patience': 15,
    'learning_rate': 0.00026,
}

test_sharpes = []
val_sharpes = []
train_sharpes = []

for run in range(n_runs):
    print(f"\n--- Run {run+1}/{n_runs} ---")
    
    # Get base config
    config = get_config('ema_sharpe', 'technical')
    config.update(best_params)
    
    # Use absolute path for data directory
    config['data_dir'] = str(DATA_DIR)
    
    # Different seed for each run
    config['seed'] = 42 + run
    
    # Train
    result = train('technical', 'PPO', 'ema_sharpe', config, verbose=False)
    
    train_sharpes.append(result['train_sharpe'])
    val_sharpes.append(result['val_sharpe'])
    test_sharpes.append(result['test_sharpe'])
    
    print(f"Test Sharpe: {result['test_sharpe']:.3f}")

# Calculate statistics
print("\n" + "="*70)
print("AGGREGATE RESULTS")
print("="*70)
print(f"Train Sharpe: {np.mean(train_sharpes):.3f} ± {np.std(train_sharpes):.3f}")
print(f"Val Sharpe:   {np.mean(val_sharpes):.3f} ± {np.std(val_sharpes):.3f}")
print(f"Test Sharpe:  {np.mean(test_sharpes):.3f} ± {np.std(test_sharpes):.3f}")
print(f"\nExpected Test Sharpe: 1.52 (base) / 1.82 (super agent)")
print(f"Min:  {np.min(test_sharpes):.3f}")
print(f"Max:  {np.max(test_sharpes):.3f}")
print("\nAll test Sharpes:", [f"{s:.3f}" for s in test_sharpes])
print("="*70)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


RUNNING 10 TIMES TO GET AVERAGE PERFORMANCE

--- Run 1/10 ---
Test Sharpe: 1.589

--- Run 2/10 ---
Test Sharpe: 1.173

--- Run 3/10 ---
Test Sharpe: 1.574

--- Run 4/10 ---
Test Sharpe: 1.140

--- Run 5/10 ---
Test Sharpe: 0.920

--- Run 6/10 ---
Test Sharpe: 0.864

--- Run 7/10 ---
Test Sharpe: 1.277

--- Run 8/10 ---
Test Sharpe: 1.389

--- Run 9/10 ---
Test Sharpe: 1.265

--- Run 10/10 ---
Test Sharpe: 1.147

AGGREGATE RESULTS
Train Sharpe: 3.623 ± 1.378
Val Sharpe:   2.486 ± 0.167
Test Sharpe:  1.234 ± 0.229

Expected Test Sharpe: 1.52 (base) / 1.82 (super agent)
Min:  0.864
Max:  1.589

All test Sharpes: ['1.589', '1.173', '1.574', '1.140', '0.920', '0.864', '1.277', '1.389', '1.265', '1.147']
